## Planning:
- Load data
- Check variable and target
- Check correlation between numeric variable
- Check output to variable impact (using KDE distribution)

In [2]:
# data processing
import pandas as pd
import numpy as np

# data visualisation
import seaborn as sns
import matplotlib.pyplot as plt

# data modeling 
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# model 
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

In [3]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
submit = pd.read_csv("sample_submission.csv")

In [4]:
train['Arrival Delay in Minutes'] = train['Arrival Delay in Minutes'].fillna(train['Arrival Delay in Minutes'].mean())

In [5]:
test['Arrival Delay in Minutes'] = test['Arrival Delay in Minutes'].fillna(test['Arrival Delay in Minutes'].mean())

In [6]:
# check correlation of numeric variable
# take all column numeric column
numeric_cols = train.drop(columns='id').select_dtypes(include='number').columns.tolist()

In [7]:
non_numeric_cols = train.select_dtypes(exclude='number').columns.tolist()

## Basic Data Process for Modeling

Data preparation - Create Dummy variable

In [8]:
# train_dummy = pd.get_dummies(train, columns=non_numeric_cols, dtype=int)
train_dummy = pd.get_dummies(train, columns=non_numeric_cols,drop_first=True, dtype=int)

In [9]:
test_dummy = pd.get_dummies(test, columns=non_numeric_cols.remove('satisfaction'),drop_first=True, dtype=int)

In [10]:
X = train_dummy.drop(columns=["id","satisfaction_True"])
y = train_dummy["satisfaction_True"]
X_test = test_dummy.drop(columns=["id"])

## Find Optimum Parameter using Optuna for XGBOOST

In [1]:
import optuna

In [11]:
SEED = 42

In [19]:
print(xgb.__version__)

3.2.0


In [22]:
# define objective using XGBoost
# Step 1: Define the Optuna objective with TimeSeriesSplit on x_train
def objective(trial):
    np.random.seed(42)
    params = {
        # fix parameter
        "objective":"binary:logistic",
        "eval_metric": "auc",
        "n_estimators": 5000,
        "tree_method": "hist",
        "enable_categorical": True,
        "random_state": SEED,
        "colsample_bytree":0.8,
        "subsample":0.8,
        "n_jobs":-1,

        # trial parameter
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2,log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 1.0,log=True),
        "reg_alpha": trial.suggest_float("alpha", 1e-8, 1.0,log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 5, 30),

        # early stopping round
        "early_stopping_rounds": 200,
    }
    
    train_idx = int(0.85 * X.shape[0])

    x_tr, x_val = X.iloc[:train_idx], X.iloc[train_idx:]
    y_tr, y_val = y.iloc[:train_idx], y.iloc[train_idx:]

    model = xgb.XGBClassifier(**params)
    model.fit(x_tr, y_tr,
              eval_set=[(x_val, y_val)],
              verbose =False)

    preds = model.predict_proba(x_val)[:,1]
    # rmse = np.sqrt(root_mean_squared_error(y_val, preds))
    score_auc = roc_auc_score(y_val, preds)

    # Save best iteration
    trial.set_user_attr("best_iteration", model.best_iteration)

    return score_auc

In [23]:
# Step 2: Run the Optuna study
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=150)

[I 2026-10-29 08:12:45,139] A new study created in memory with name: no-name-24a0c988-0d61-4284-b0ed-8008a96c2bf7
[I 2026-10-29 08:14:16,435] Trial 0 finished with value: 0.9590894203767645 and parameters: {'learning_rate': 0.030710573677773714, 'max_depth': 8, 'reg_lambda': 0.007177141927992002, 'alpha': 0.0006155564318973012, 'min_child_weight': 9}. Best is trial 0 with value: 0.9590894203767645.
[I 2026-10-29 08:19:22,318] Trial 1 finished with value: 0.9572808866437394 and parameters: {'learning_rate': 0.015957084694148364, 'max_depth': 3, 'reg_lambda': 0.08499808989182997, 'alpha': 0.0006440507553993703, 'min_child_weight': 23}. Best is trial 0 with value: 0.9590894203767645.
[I 2026-10-29 08:22:58,406] Trial 2 finished with value: 0.9591734451859021 and parameters: {'learning_rate': 0.010636066512540286, 'max_depth': 8, 'reg_lambda': 0.04566054873446119, 'alpha': 4.997040685255803e-07, 'min_child_weight': 9}. Best is trial 2 with value: 0.9591734451859021.
[I 2026-10-29 08:28:24,

In [24]:
# Step 4: Train final model on full x_train with best parameters
best_params = study.best_params

In [29]:
best_trial = study.best_trial
best_params_from_optuna = study.best_trial.params

In [31]:
print("Best Value (AUC):", study.best_value)
print("Best Params:", best_params)
print("Best iteration (trees):", best_trial.user_attrs["best_iteration"])
print("Best param from optuna:", best_params_from_optuna)

Best Value (AUC): 0.9593679653113212
Best Params: {'learning_rate': 0.023621412607945612, 'max_depth': 8, 'reg_lambda': 0.15767822496360231, 'alpha': 0.3574854835852679, 'min_child_weight': 29}
Best iteration (trees): 1444
Best param from optuna: {'learning_rate': 0.023621412607945612, 'max_depth': 8, 'reg_lambda': 0.15767822496360231, 'alpha': 0.3574854835852679, 'min_child_weight': 29}


In [32]:
final_params = {
    # fix parameter
        "objective":"binary:logistic",
        "eval_metric": "auc",
        "n_estimators": 5000,
        "tree_method": "hist",
        "enable_categorical": True,
        "random_state": SEED,
        "colsample_bytree":0.8,
        "subsample":0.8,
        "n_jobs":-1,
        # early stopping round
        "early_stopping_rounds": 200,

        **best_params
}

## Model using XGboost 5 fold CV using Best Parameter Optuna

In [33]:
SEED = 42
N_FOLDS = 5
TARGET = 'satisfaction_True'

In [65]:
# Stratified KFold cross validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))

In [66]:
def run_cv(fit_predict_fn, name):
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        va_pred, te_pred = fit_predict_fn(X_tr, y_tr, X_va, y_va)
        oof[va_idx] = va_pred
        test_pred += te_pred / N_FOLDS
        auc = roc_auc_score(y_va, va_pred)
        scores.append(auc)
        print(f'  fold {fold}: AUC = {auc:.5f}')
    total = roc_auc_score(y, oof)
    print(f'{name} | OOF AUC = {total:.5f} (mean {np.mean(scores):.5f} ± {np.std(scores):.5f})')
    return oof, test_pred, total

In [35]:
def fit_xgb(X_tr, y_tr, X_va, y_va):
    model = xgb.XGBClassifier(**final_params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return (model.predict_proba(X_va)[:, 1],
            model.predict_proba(X_test)[:, 1])

In [36]:
oof_xgb, test_xgb, auc_xgb = run_cv(fit_xgb, 'XGBoost')

  fold 0: AUC = 0.95894
  fold 1: AUC = 0.95784
  fold 2: AUC = 0.95963
  fold 3: AUC = 0.95880
  fold 4: AUC = 0.95907
XGBoost | OOF AUC = 0.95885 (mean 0.95886 ± 0.00058)


In [37]:
def save_sub(pred, name):
    sub = submit.copy()
    sub['satisfaction'] = pred
    sub.to_csv(name, index=False)
    print('saving file:', name)

In [39]:
save_sub(test_xgb,'sub_xgb_6_5cv_optuna.csv')

saving file: sub_xgb_6_5cv_optuna.csv


In [50]:
train.head()

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False


In [57]:
oof_xgb_df = train[['id']].copy()

In [58]:
oof_xgb_df.loc[:,'oof_xgb_6_5cv_optuna'] = oof_xgb

In [59]:
oof_xgb_df.to_csv('oof_xgb_6_5cv_optuna.csv')

In [60]:
auc_xgb
# 0.9588476316831602

0.9588476316831602

### XGB 20 Folds CV

In [67]:
SEED = 42
N_FOLDS = 20
TARGET = 'satisfaction_True'

In [68]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))

In [69]:
oof_xgb, test_xgb, auc_xgb = run_cv(fit_xgb, 'XGBoost')

  fold 0: AUC = 0.95953
  fold 1: AUC = 0.95872
  fold 2: AUC = 0.95943
  fold 3: AUC = 0.95868
  fold 4: AUC = 0.95645
  fold 5: AUC = 0.95820
  fold 6: AUC = 0.95569
  fold 7: AUC = 0.96136
  fold 8: AUC = 0.95849
  fold 9: AUC = 0.96078
  fold 10: AUC = 0.95871
  fold 11: AUC = 0.96078
  fold 12: AUC = 0.95823
  fold 13: AUC = 0.95788
  fold 14: AUC = 0.95936
  fold 15: AUC = 0.96063
  fold 16: AUC = 0.96022
  fold 17: AUC = 0.95821
  fold 18: AUC = 0.95921
  fold 19: AUC = 0.95952
XGBoost | OOF AUC = 0.95899 (mean 0.95900 ± 0.00138)


In [70]:
# submission
save_sub(test_xgb,'sub_xgb_7_20cv_optuna.csv')

saving file: sub_xgb_7_20cv_optuna.csv


In [71]:
oof_xgb_df = train[['id']].copy()
oof_xgb_df.loc[:,'oof_xgb_7_20cv_optuna'] = oof_xgb
oof_xgb_df.to_csv('oof_xgb_7_20cv_optuna.csv')

In [72]:
auc_xgb

0.9589902806172532